# UniSEM · Segment your EM data

Run the first-stage **EMNeuron generalist** on a grayscale image or image stack. This notebook uses your own Colab runtime; uploaded images are not sent to the project website or model repository.

1. Prepare the environment and model.
2. Upload an image.
3. Run segmentation.
4. Inspect and download the result.

For practical inference, select **Runtime → Change runtime type → GPU** before starting. Run the cells in order. The model produces **independent 2D instances per slice**, without reconstructing 3D identities.

[Project](https://oxidationreduction.github.io/UniSEM-ISBI/) · [Code](https://github.com/oxidationreduction/UniSEM-ISBI) · [Model card](https://modelscope.cn/models/dreamBuilder/UniSEM)

## 1. Prepare the environment

This cell installs UniSEM in an isolated Python 3.10 environment, leaving the notebook kernel for file upload and visualization. Dependencies and the exact code/model revisions are fixed. The first download includes the model and its matched random forest.

In [ ]:
import hashlib
import io
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tarfile
import time
from urllib.request import urlopen

CODE_REVISION = "6645bc3725b202c454076899ba9d7bcbdc67b484"
MODEL_REVISION = "c074500d3426dfe4d43d47fb237775c2e2c788d5"

for revision in (CODE_REVISION,):
    if len(revision) != 40 or any(c not in "0123456789abcdef" for c in revision):
        raise ValueError("The published notebook must contain fixed 40-character code and model revisions.")

WORK = Path("/content/unisem-demo")
WORK.mkdir(parents=True, exist_ok=True)
REPO = WORK / "source"
if not REPO.exists():
    subprocess.run(["git", "clone", "--no-checkout", "https://github.com/oxidationreduction/UniSEM-ISBI.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_REVISION], check=True)
else:
    actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    if actual != CODE_REVISION:
        raise ValueError("An existing source directory has a different revision. Start a fresh runtime for this notebook version.")

# Visualization packages run in the notebook kernel; model dependencies stay isolated.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "ipywidgets", "tifffile", "h5py", "pillow"], check=True)
MAMBA = WORK / "micromamba"
if not MAMBA.exists():
    with urlopen("https://micro.mamba.pm/api/micromamba/linux-64/2.0.5", timeout=120) as response:
        archive_bytes = response.read()
    with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:bz2") as archive:
        binary = archive.extractfile("bin/micromamba")
        if binary is None:
            raise RuntimeError("The official micromamba archive did not contain its executable.")
        MAMBA.write_bytes(binary.read())
    MAMBA.chmod(0o755)

ENVIRONMENT = WORK / "environment"
COMMAND_ENV = dict(os.environ, MAMBA_ROOT_PREFIX=str(WORK / "mamba-root"), MPLBACKEND="Agg")
if not (ENVIRONMENT / "bin/python").exists():
    subprocess.run([str(MAMBA), "create", "-y", "--strict-channel-priority", "-p", str(ENVIRONMENT), "-f", str(REPO / "environment.colab.yml")], env=COMMAND_ENV, check=True)
PYTHON = ENVIRONMENT / "bin/python"
subprocess.run([str(PYTHON), "-m", "pip", "install", "--constraint", str(REPO / "constraints.txt"), "-e", str(REPO)], env=COMMAND_ENV, check=True)
subprocess.run([str(PYTHON), "-c", "import torch, micro_sam, elf.segmentation, vigra, nifty, affogato; from micro_sam.instance_segmentation import get_unetr; print('Model dependencies imported successfully')"], env=COMMAND_ENV, check=True)
DEVICE = subprocess.check_output([str(PYTHON), "-c", "import torch; print('cuda' if torch.cuda.is_available() else 'cpu')"], env=COMMAND_ENV, text=True).strip()

print("Environment ready. Device:", DEVICE)
print("Continue with the model-download cell below.")


### Download the model

Download and verify the first-stage model and its matching RF. This cell uses the immutable model revision recorded above.

In [ ]:
sys.path.insert(0, str(REPO / "src"))
from unisem.demo import DEMO_ASSETS, download_demo_assets, export_results, file_sha256, load_input, overlay_image, select_input
ASSETS = download_demo_assets(WORK / "models", MODEL_REVISION, provider="modelscope")
print("Ready. Device:", DEVICE)
print("Code revision:", CODE_REVISION)
print("Model revision:", MODEL_REVISION)
print("Both asset hashes verified; only this published RF will be loaded.")

## 2. Upload an image

Upload a **grayscale TIFF** (YX image or ZYX stack) or an **HDF5** file. Select an HDF5 dataset key if needed. Pixel values and image resolution are preserved; inference applies the existing per-slice intensity normalization.

For an ambiguous HDF5 shape ending in 3 or 4, select ZYX only if it really is a grayscale stack. RGB/channel TIFFs are rejected. The optional bundled example is a WaferEM display crop, shown under its [dataset terms](https://huggingface.co/datasets/cyd0806/wafer_EM); it is not new performance evidence for the generalist model.

In [ ]:
from google.colab import files, output
output.enable_custom_widget_manager()
from IPython.display import display, clear_output
import ipywidgets as widgets
import numpy as np
from PIL import Image
import tifffile

UPLOADS = WORK / "uploads"
UPLOADS.mkdir(exist_ok=True)
input_state = {}
file_select = widgets.Dropdown(description="Input:", options=[] , layout=widgets.Layout(width="95%"))
key_field = widgets.Combobox(value="main", options=["main"], description="HDF5 key:", ensure_option=False, layout=widgets.Layout(width="95%"))
axes_select = widgets.Dropdown(description="Axes:", options=[("Detect grayscale dimensions", "auto"), ("2D image · YX", "YX"), ("Grayscale stack · ZYX", "ZYX")])
load_button = widgets.Button(description="Inspect selected input", button_style="primary")
upload_button = widgets.Button(description="Upload TIFF / HDF5")
example_button = widgets.Button(description="Use bundled example")
input_log = widgets.Output()
slice_select = widgets.IntSlider(value=0, min=0, max=0, description="Slice:", continuous_update=False)
scope_select = widgets.ToggleButtons(description="Run:", options=[("Selected slice", "slice"), ("Entire stack", "stack")], value="slice")

def refresh_files():
    file_select.options = [(p.name, str(p)) for p in sorted(UPLOADS.iterdir()) if p.suffix.lower() in {".tif", ".tiff", ".h5", ".hdf5"}]

def refresh_hdf5_keys(_=None):
    if file_select.value and Path(file_select.value).suffix.lower() in {".h5", ".hdf5"}:
        import h5py
        try:
            names = []
            with h5py.File(file_select.value, "r") as handle:
                handle.visititems(lambda name, item: names.append(name) if isinstance(item, h5py.Dataset) else None)
            key_field.options = names
            key_field.value = "main" if "main" in names else (names[0] if names else "main")
        except Exception as error:
            with input_log:
                print(type(error).__name__ + ": " + str(error))
    else:
        key_field.options = ["main"]
        key_field.value = "main"


def upload_clicked(_):
    input_state.clear()
    with input_log:
        clear_output(wait=True)
        uploaded = files.upload()
        for name, content in uploaded.items():
            filename = Path(name).name
            if Path(filename).suffix.lower() not in {".tif", ".tiff", ".h5", ".hdf5"}:
                print("Skipped unsupported file:", filename)
                continue
            (UPLOADS / filename).write_bytes(content)
        refresh_files()
        print("Choose a file and inspect it before running.")

def example_clicked(_):
    input_state.clear()
    with input_log:
        clear_output(wait=True)
        try:
            url = f"https://raw.githubusercontent.com/oxidationreduction/UniSEM-ISBI/{CODE_REVISION}/site/assets/crops/typical_improvement_z080_raw.png"
            with urlopen(url, timeout=60) as response:
                image = np.asarray(Image.open(io.BytesIO(response.read())).convert("L"))
            tifffile.imwrite(UPLOADS / "waferem-display-crop.tif", image, photometric="minisblack")
            refresh_files()
            file_select.value = str(UPLOADS / "waferem-display-crop.tif")
            print("Example selected. Inspect it to continue.")
        except Exception as error:
            print(type(error).__name__ + ": " + str(error))

def inspect_clicked(_):
    input_state.clear()
    with input_log:
        clear_output(wait=True)
        try:
            if file_select.value is None:
                raise ValueError("Upload or select an image first.")
            raw, metadata = load_input(file_select.value, key=key_field.value, axes=axes_select.value)
            input_state.update(raw=raw, metadata=metadata)
            slice_select.max = len(raw) - 1 if raw.ndim == 3 else 0
            slice_select.value = 0
            slice_select.disabled = raw.ndim == 2
            print(json.dumps(metadata, indent=2))
            display(Image.fromarray(overlay_image(raw if raw.ndim == 2 else raw[0], np.zeros(raw.shape[-2:], dtype=np.uint32), opacity=0)))
        except Exception as error:
            print(type(error).__name__ + ": " + str(error))

def invalidate_inspection(_):
    input_state.clear()

for control in (file_select, key_field, axes_select):
    control.observe(invalidate_inspection, names="value")
file_select.observe(refresh_hdf5_keys, names="value")
upload_button.on_click(upload_clicked)
example_button.on_click(example_clicked)
load_button.on_click(inspect_clicked)
refresh_files()
display(widgets.HBox([upload_button, example_button]), file_select, key_field, axes_select, load_button, input_log)

## 3. Run segmentation

The first-stage EMNeuron model is used for every input. It does not retrain on your upload. Select one slice for a quick look, or process the full stack. The pipeline uses the published 256/384/512-window predictions, watershed fragments and matched 55-feature random forest.

Run time depends on your image and runtime. The log below reports the actual command outcome; a failed run never creates a success preview.

In [ ]:
run_button = widgets.Button(description="Run UniSEM", button_style="primary", icon="play")
run_log = widgets.Output()
result_state = {}
result_panel = widgets.Output()

def run_clicked(_):
    result_state.clear()
    result_panel.clear_output()
    run_button.disabled = True
    with run_log:
        clear_output(wait=True)
        try:
            if "raw" not in input_state:
                raise ValueError("Inspect a valid input in step 2 first.")
            selected = select_input(input_state["raw"], scope_select.value, slice_select.value)
            run_directory = WORK / "runs" / time.strftime("%Y%m%d-%H%M%S")
            run_directory.mkdir(parents=True, exist_ok=False)
            input_file = run_directory / "input.tif"
            tifffile.imwrite(input_file, selected, photometric="minisblack", metadata={"axes": "YX" if selected.ndim == 2 else "ZYX"})
            result_file = run_directory / "prediction.h5"
            # Recheck identities immediately before strict model/RF loading.
            for filename, expected in DEMO_ASSETS.items():
                if file_sha256(ASSETS[filename]) != expected:
                    raise ValueError("A model asset changed after setup; rerun step 1.")
            command = [str(ENVIRONMENT / "bin/unisem"), "infer", "--dataset", "emneuron", "--config", str(REPO / "configs/paper_protocol.json"), "--checkpoint", str(ASSETS["unisem_stage1_emneuron_vitl.pt"]), "--rag-checkpoint", str(ASSETS["unisem_stage1_emneuron_full55_rf.joblib"]), "--input", str(input_file), "--output", str(result_file), "--device", DEVICE]
            print("Running on", DEVICE, "with shape", selected.shape, flush=True)
            started = time.perf_counter()
            completed = subprocess.run(command, env=COMMAND_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            print(completed.stdout or "", end="", flush=True)
            completed.check_returncode()
            info = {"input": input_state["metadata"], "scope": scope_select.value, "selected_input_slice": slice_select.value if selected.ndim == 2 and input_state["raw"].ndim == 3 else None, "code_revision": CODE_REVISION, "model_repository": "dreamBuilder/UniSEM", "model_provider": "modelscope", "model_revision": MODEL_REVISION, "asset_sha256": DEMO_ASSETS, "device": DEVICE, "elapsed_seconds": round(time.perf_counter() - started, 3)}
            exported = export_results(selected, result_file, run_directory / "exports", info)
            import h5py
            with h5py.File(result_file, "r") as handle:
                labels = np.asarray(handle["final_merged_segmentation"])
            result_state.update(raw=selected, labels=labels, exported=exported)
            print("Finished. Result shape:", labels.shape)
            print("Run step 4 to inspect and download your actual prediction.")
        except Exception as error:
            print(type(error).__name__ + ": " + str(error))
            print("No new result is available. Resolve this error before continuing.")
        finally:
            run_button.disabled = False

run_button.on_click(run_clicked)
display(slice_select, scope_select, run_button, run_log, result_panel)

## 4. Inspect and download

Run this cell after segmentation finishes. Use the slice and opacity controls to inspect the actual predictions. Left: raw EM. Right: model instances or boundaries. Instance colors are for display; the same number in different slices does not establish 3D identity.

The download contains `instances.tif` (uint32 labels), `prediction.h5` (dense maps and final labels), `preview.png` and `run.json` (versions, settings and hashes).

In [ ]:
if not result_state:
    raise ValueError("Complete a successful segmentation in step 3 before opening the result viewer.")

output_raw = result_state["raw"]
output_labels = result_state["labels"]
output_slice = widgets.IntSlider(value=0, min=0, max=len(output_raw) - 1 if output_raw.ndim == 3 else 0, description="Slice:", continuous_update=False)
output_slice.disabled = output_raw.ndim == 2
opacity_control = widgets.FloatSlider(value=0.65, min=0, max=1, step=0.05, description="Opacity:", continuous_update=False)
mode_control = widgets.ToggleButtons(options=[("Boundaries", "boundaries"), ("Instances", "instances")], value="boundaries")
preview = widgets.Output()
download_button = widgets.Button(description="Download results ZIP", button_style="primary", icon="download")

def update_preview(_=None):
    with preview:
        clear_output(wait=True)
        raw = output_raw if output_raw.ndim == 2 else output_raw[output_slice.value]
        labels = output_labels if output_labels.ndim == 2 else output_labels[output_slice.value]
        empty = np.zeros(raw.shape, dtype=np.uint32)
        image = np.concatenate((overlay_image(raw, empty, opacity=0), overlay_image(raw, labels, opacity=opacity_control.value, mode=mode_control.value)), axis=1)
        display(Image.fromarray(image))

for control in (output_slice, opacity_control, mode_control):
    control.observe(update_preview, names="value")
download_button.on_click(lambda _: files.download(str(result_state["exported"]["download"])))
with result_panel:
    clear_output(wait=True)
    display(output_slice, opacity_control, mode_control, preview, download_button)
    update_preview()

### Model and data notes

This demonstration uses the first-stage EMNeuron model and its published full55 RF, not a WaferEM-specialized model. New acquisition conditions can change performance. Inspect the result on your own data before downstream analysis.

The code, model and source datasets have separate attribution and terms; see the [model card](https://modelscope.cn/models/dreamBuilder/UniSEM) and [third-party notices](https://github.com/oxidationreduction/UniSEM-ISBI/blob/main/THIRD_PARTY_NOTICES.md). Save your downloaded outputs before the Colab runtime ends.